Driver notebook orchestrating the MIMIC-IV deterioration prediction pipeline with FSDP/DDP distributed training benchmarks.

In [ ]:
import os, sys
sys.path.insert(0, os.getcwd())

from snowflake.snowpark.context import get_active_session
session = get_active_session()

import deterioration_config as cfg
session.sql('USE WAREHOUSE DEIDENTIFIED_LOAD_MEDIUM').collect()
session.sql(f'USE SCHEMA {cfg.OUT_SCHEMA}').collect()
print('role     :', session.get_current_role())
print('warehouse:', session.get_current_warehouse())
print('schema   :', session.get_fully_qualified_current_schema())
print('outputs  :', cfg.TS_TABLE, '|', cfg.NOTES_TABLE, '|', cfg.MULTIMODAL_TABLE)
print('features :', len(cfg.FEATURE_NAMES), 'x', cfg.SEQ_LEN, '=', len(cfg.FEATURE_NAMES)*cfg.SEQ_LEN, 'flat columns')

In [ ]:
with open('phase0_load_radiology_notes.sql') as f:
    raw = f.read()
sql_no_comments = '\n'.join(l for l in raw.splitlines() if not l.strip().startswith('--'))
stmts = [s.strip() for s in sql_no_comments.split(';') if s.strip()]
for s in stmts:
    print('>>', s.strip().splitlines()[0][:80])
    print(session.sql(s).collect())

In [ ]:
import importlib, preprocess_deterioration_snowpark as prep
importlib.reload(prep)
scaler = prep.run(session)

Feature columns in the final table are imputed (missing → 0) so they are non-null

In [ ]:
import pandas as pd, numpy as np
pd.set_option('display.max_rows', 40)
FEATS = cfg.FEATURE_NAMES
H = cfg.SEQ_LEN
flat = cfg.all_flat_columns()
TS = cfg.TS_TABLE
RAW = TS + '_RAW'

# 1) Row counts + label prevalence per split
counts = session.sql(f"""
  SELECT split, COUNT(*) n, SUM(label) positives, AVG(label) prevalence,
         COUNT(DISTINCT hadm_id) admissions
  FROM {TS} GROUP BY split ORDER BY split
""").to_pandas()
total = counts['N'].sum()
print('=== Row counts ===')
print(counts.to_string(index=False))
print(f'TOTAL rows: {total:,} | feature cols: {len(flat)} | total cols incl metadata: {len(flat)+6}')

In [ ]:
# 2) Metadata null + range checks (these SHOULD be non-null)
meta = session.sql(f"""
  SELECT
    SUM(IFF(label IS NULL,1,0))         AS label_nulls,
    SUM(IFF(sample_weight IS NULL,1,0)) AS weight_nulls,
    SUM(IFF(split IS NULL,1,0))         AS split_nulls,
    SUM(IFF(hadm_id IS NULL,1,0))       AS hadm_nulls,
    SUM(IFF(tsp IS NULL,1,0))           AS tsp_nulls,
    MIN(sample_weight) AS w_min, MAX(sample_weight) AS w_max, AVG(sample_weight) AS w_mean,
    AVG(IFF(sample_weight=0,1,0)) AS frac_weight_zero,
    AVG(IFF(sample_weight<=0.0161,1,0)) AS frac_weight_floor
  FROM {TS}
""").to_pandas()
print('=== Metadata null & sample_weight checks ===')
print(meta.T.to_string(header=False))

In [ ]:
# 3) TRUE pre-imputation missingness per feature, from the _RAW staging table.
# null-rate(feature) = (sum of NULLs across its 48 hourly columns) / (48 * rows)
try:
    n_raw = session.sql(f'SELECT COUNT(*) c FROM {RAW}').collect()[0]['C']
    per_feat = ',\n'.join(
        '(' + '+'.join(f'IFF({cfg.flat_col(f,h)} IS NULL,1,0)' for h in range(H)) +
        f')/{float(H)} AS \"{f}\"'
        for f in FEATS
    )
    miss = session.sql(f'SELECT AVG(CASE WHEN TRUE THEN x END) FROM (SELECT 1 x) LIMIT 0')  # noop placeholder
    raw_miss = session.sql(f'SELECT ' + ', '.join(
        'AVG(' + '+'.join(f'IFF({cfg.flat_col(f,h)} IS NULL,1,0)' for h in range(H)) + f')/{float(H)} AS \"{f}\"'
        for f in FEATS) + f' FROM {RAW}').to_pandas()
    s = raw_miss.T[0].sort_values()
    print(f'=== True missingness per feature (RAW, {n_raw:,} rows) ===')
    print('overall mean missing fraction: %.3f' % s.mean())
    print('\n-- 8 best observed (lowest missing) --')
    print((s.head(8)).to_string())
    print('\n-- 8 most missing --')
    print((s.tail(8)).to_string())
except Exception as e:
    print('RAW staging table not available in this session (', e, ')')
    print('Skipping pre-imputation analysis; see final-table zero-rate below.')

In [ ]:
# 4) Confirm final table has NO nulls in feature columns (imputation worked),
#    and report per-feature imputed-zero rate (should track RAW missingness).
null_check = session.sql('SELECT ' + ' + '.join(
    f'IFF(MAX(IFF({c} IS NULL,1,0))=1,1,0)' for c in flat[:200]) +
    f' AS any_null_in_first_200 FROM {TS}').to_pandas()
print('Feature NULLs in first 200 cols (expect 0):', int(null_check.iloc[0,0]))

zero_rate = session.sql('SELECT ' + ', '.join(
    'AVG(' + '+'.join(f'IFF({cfg.flat_col(f,h)}=0,1,0)' for h in range(H)) + f')/{float(H)} AS \"{f}\"'
    for f in FEATS) + f' FROM {TS}').to_pandas()
z = zero_rate.T[0].sort_values()
print('\n=== Imputed-zero rate per feature (final table) ===')
print('overall mean zero fraction: %.3f' % z.mean())
print(z.to_string())

## BioBERT note embeddings (GPU)
Downloads BioBERT from stage, embeds radiology notes, then joins with each observation 

In [ ]:
import importlib, embed_notes_biobert as emb
importlib.reload(emb)
# Set skip_embedding=True to rebuild w/obs-window join from existing table
emb.run(session, batch_rows=20000, gpu_batch=64, skip_embedding=False)

## Pre-materialize the multimodal table, LEFT JOIN aligns notes onto the TS table

In [ ]:
import importlib, build_multimodal_training_table as mm
importlib.reload(mm)
mm.run(session)

## RUS computation (GPU)
Computes population-level Redundancy/Uniqueness/Synergy for labs_vitals vs notes modalities. Saves the `.npy` consumed by training.

In [ ]:
import importlib, deterioration_rus_snowflake as rus
importlib.reload(rus)
rus_path = rus.main(session, num_subsample=5000, seq_len=48, num_lags=6, sequence_pooling='timestep')
print('RUS saved to:', rus_path)

## Train multimodal TRUS-MoE (GPU)


In [ ]:
import importlib, train_deterioration_snowflake as T
importlib.reload(T)
best_auroc = T.main(
    session,
    ts_only=False,
    experiment_name='DETERIORATION_TRUS_MOE',
    rus_path=rus_path,
    epochs=10, batch_size=256, chunk_rows=20000,
)
print('Best val AUROC:', best_auroc)

## XGBoost baseline

In [ ]:
import time, datetime
while True:
    result = session.sql("SELECT CURRENT_TIMESTAMP() AS ts").collect()
    print(f"[{datetime.datetime.now():%H:%M:%S}] session alive — {result[0]['TS']}")
    time.sleep(300)

## Distributed FSDP training

In [ ]:
import importlib, train_deterioration_distributed as TD
importlib.reload(TD)
response = TD.launch(
    session,
    ts_only=False,            
    experiment_name='DETERIORATION_FSDP_TEST_2GPU',
    d_model=1024, d_ff=4096, nhead=16,
    moe_num_experts=8, moe_expert_hidden_dim=1024,
    num_encoder_layers=4, num_moe_layers=2,
    modality_encoder_layers=3,
    epochs=1, batch_size=512,
    num_gpus=4, num_nodes=1,
)

In [ ]:
!nvidia-smi

### DDP benchmark 

In [ ]:
import importlib, train_deterioration_ddp as TD_DDP
importlib.reload(TD_DDP)

response_ddp = TD_DDP.launch(
    session,
    ts_only=False,
    experiment_name='DETERIORATION_DDP_TEST_4GPU',
    d_model=1024, d_ff=4096, nhead=16,
    moe_num_experts=8, moe_expert_hidden_dim=1024,
    num_encoder_layers=4, num_moe_layers=2,
    modality_encoder_layers=3,
    epochs=1, batch_size=512,
    num_gpus=4, num_nodes=1,
)

In [ ]:
import importlib, benchmark_dataloader as BD
importlib.reload(BD)

response_bench = BD.launch(
    session,
    num_gpus=4, num_nodes=1,
    batch_size=512,
    max_batches=200,
    warmup_batches=5,
)